In [33]:

from rockyclickup.wrapper import Session
from rockyclickup.utils import response_to_dataframe

In [34]:
rcu = Session()

In [35]:
spaces = rcu.get_spaces()

In [36]:
shared_services_space = [c for c in spaces if "shared" in c['name'].lower()][0]
shared_services_folders = rcu.get_folders(shared_services_space.get("id"))
data_processing_folder = [c for c in shared_services_folders if "data" in c.get("name").lower()][0]
data_processing_lists = rcu.get_lists(data_processing_folder.get("id"))
gateway_list = [c for c in data_processing_lists if "gateway" in c.get("name").lower()][0]
gateway_list_id = gateway_list.get("id")

In [37]:
gateway_list_items = rcu.get_full_list(list_id=gateway_list_id)

In [40]:
len(gateway_list_items)

8240

In [ ]:
len(gateway_list_dataframe)

# 10573 - 8240 = 2333


10573

In [55]:
gateway_list_dataframe = response_to_dataframe(gateway_list_items)

Field not in config.db:	2947ad1a-9b12-4800-82c4-138325278c8c	`debug_data_id`
Field not in config.db:	032df49e-6d2b-47ee-b40a-4794650c488a	`debug_assignee_id`


In [56]:
len(gateway_list_dataframe)

10573

In [58]:
[c for c in gateway_list_dataframe if "date" in c.lower()]

['date_created',
 'date_updated',
 'date_closed',
 'date_done',
 'due_date',
 'start_date',
 'file_date']

In [ ]:
import pandas as pd


def to_local_naive(series: pd.Series) -> pd.Series:
    """Coerce a column of mixed tz-aware / tz-naive python datetimes into one
    naive, local wall-clock datetime64 column so it can be sorted and compared.

    rockyclickup's milliseconds_to_datetime returns tz-aware datetimes normally but a
    *naive* date for ClickUp's 4am date-only default, so the raw columns are a mix of
    both and pandas refuses to sort them ("can't compare offset-naive and offset-aware").
    """

    def _strip(value):
        if pd.isna(value):
            return pd.NaT
        if getattr(value, "tzinfo", None) is not None:
            return value.astimezone().replace(tzinfo=None)  # local wall clock, tz dropped
        return value

    return pd.to_datetime(series.map(_strip))


# only the columns the duplicate analysis relies on; add more here if you sort on them
for col in ["file_date", "date_created"]:
    gateway_list_dataframe[col] = to_local_naive(gateway_list_dataframe[col])

gateway_list_dataframe[["file_date", "date_created"]].dtypes

In [ ]:
# file_date is a date-only field, so any value with a time component is a tz artefact
# (a value that dodged the library's 4am rule). if this is > 0, consider adding
# gateway_list_dataframe["file_date"] = gateway_list_dataframe["file_date"].dt.normalize()
# so those rows match their midnight siblings in the duplicate key
file_date = gateway_list_dataframe["file_date"]
(file_date.notna() & (file_date != file_date.dt.normalize())).sum()

In [ ]:
duplicate_rows = gateway_list_dataframe[gateway_list_dataframe.duplicated(subset=["file_date", "name"])]

# I need another dataframe showing the duplicate rows, but i want to keep the rows that have an older date_created date
# that way i could then get a list of the duplicate rows created AFTER the first instance

# i also want to get a dataframe of file names with a nother column of how many times that name appears in the original dataframe



In [62]:
len(duplicate_rows)

2333

In [ ]:
# every row that shares a (file_date, name) with at least one other row,
# ordered so the earliest-created task in each group comes first
all_duplicate_rows = gateway_list_dataframe[
    gateway_list_dataframe.duplicated(subset=["file_date", "name"], keep=False)
].sort_values(["name", "file_date", "date_created"])

len(all_duplicate_rows)

In [ ]:
# sort oldest -> newest so keep="first" always keeps the earliest-created row in each group
sorted_by_created = gateway_list_dataframe.sort_values("date_created")

in_duplicate_group = sorted_by_created.duplicated(subset=["file_date", "name"], keep=False)
created_after_first = sorted_by_created.duplicated(subset=["file_date", "name"], keep="first")

# the original task (oldest date_created) for each duplicated (file_date, name)
first_instances = sorted_by_created[in_duplicate_group & ~created_after_first]

# every duplicate created AFTER that first instance -- should equal len(duplicate_rows)
later_duplicates = sorted_by_created[created_after_first]

len(first_instances), len(later_duplicates)

In [ ]:
# the later-created duplicates, trimmed to the columns that matter for eyeballing
later_duplicates[["id", "name", "file_date", "date_created"]]

In [30]:
# task ids to delete: every duplicate (same name + file_date) except the first-created one
task_ids_to_delete = later_duplicates["id"].tolist()

# sanity checks: nothing we're keeping is in the delete list, and no id is listed twice
assert not set(task_ids_to_delete) & set(first_instances["id"])
assert len(task_ids_to_delete) == len(set(task_ids_to_delete))

print(f"{len(task_ids_to_delete)} tasks to delete")
task_ids_to_delete

2333 tasks to delete


['868kwj2g0',
 '868kwj2n0',
 '868kwja1t',
 '868kwjabr',
 '868kwjafk',
 '868kwjt90',
 '868kwk8ef',
 '868kwkfu2',
 '868kwkfhz',
 '868kwkvm6',
 '868kwkwzc',
 '868kwkx4q',
 '868kwkxex',
 '868kwkyfb',
 '868kwkydc',
 '868kwkyxr',
 '868kwkyw6',
 '868kwkyvr',
 '868kwkytj',
 '868kwkyru',
 '868kwkyqc',
 '868kwkypv',
 '868kwkyh8',
 '868kwkyud',
 '868kwm7w1',
 '868kwmdxu',
 '868kwmdz6',
 '868kwmgqr',
 '868kwmymd',
 '868kwmyu1',
 '868kwpjmx',
 '868kwq2k1',
 '868kwq2vh',
 '868kwq32p',
 '868kwqeza',
 '868kwve1k',
 '868kwvk5h',
 '868kwvwuj',
 '868kwvz9q',
 '868kwwbr8',
 '868kwwcd3',
 '868kwwdha',
 '868kwwj9d',
 '868kwwyex',
 '868kwxecr',
 '868kwxrj1',
 '868kwymxg',
 '868kwyvbz',
 '868kwz00z',
 '868kwz4fz',
 '868kx215j',
 '868kx32p0',
 '868kx3ur2',
 '868kx3xmf',
 '868kx47ee',
 '868kx47jt',
 '868kx4g01',
 '868kx4j34',
 '868kx4p9r',
 '868kx4qmg',
 '868kx4xka',
 '868kx4xhk',
 '868kx4x9z',
 '868kx4xft',
 '868kx4xen',
 '868kx4xfz',
 '868kx4y7q',
 '868kx4y73',
 '868kx4y5q',
 '868kx4y58',
 '868kx4y4h',
 '868k

In [ ]:
# how many times each file name appears in the original dataframe
name_counts = (
    gateway_list_dataframe.groupby("name", dropna=False)
    .size()
    .reset_index(name="occurrences")
    .sort_values("occurrences", ascending=False, ignore_index=True)
)

name_counts

In [ ]:
# only the names that show up more than once
repeated_names = name_counts[name_counts["occurrences"] > 1]

repeated_names

In [ ]:
gateway_list_dataframe

# 10753

,id,custom_id,custom_item_id,name,text_content,description,date_created,date_updated,date_closed,date_done,archived,assignees,group_assignees,watchers,checklists,tags,parent,top_level_parent,priority,due_date,start_date,points,time_estimate,dependencies,linked_tasks,locations,team_id,url,permission_level,status.status,status.id,status.color,status.type,status.orderindex,creator.id,creator.username,creator.color,creator.email,creator.profilePicture,list.id,list.name,list.access,notes,am,ftp_user,ftp_directory,review_status,last_reviewed,ftp_filename,received,file_category,file_date,last_emailed,debug_data_id,debug_assignee_id
0,868m73b7c,None,0,HSA Contributions Pay Date 09.18.2026.xlsx,,,2026-09-19 10:17:00-06:00,2026-09-19 10:17:00-06:00,None,NaT,False,[],[],[],[],[],None,None,None,None,None,None,None,[],[],[],9011096643,https://app.clickup.com/t/868m73b7c,create,new,sc901114339591_BKPkQ4Jh,#3db88b,open,0,75481693,rmrcloud,#5d4037,rmrcloud@rmrbenefits.com,https://attachments.clickup.com/profilePicture...,901114339591,Data Inbox [GATEWAY],True,None,NaN,None,Clients/GCommerce Solutions-RMRGCOM/Files for ...,NaN,None,HSA Contributions Pay Date 09.18.2026.xlsx,2026-09-18 23:00:00-06:00,NDT,2026-09-19 15:17:00-06:00,None,8687ad80x,None
1,868m73b3t,None,0,TEST_WestsideWomenCareRMRWWC_Contribution_2026...,,,2026-09-19 10:15:00-06:00,2026-09-19 10:15:00-06:00,None,NaT,False,[],[],[],[],[],None,None,None,None,None,None,None,[],[],[],9011096643,https://app.clickup.com/t/868m73b3t,create,new,sc901114339591_BKPkQ4Jh,#3db88b,open,0,75481693,rmrcloud,#5d4037,rmrcloud@rmrbenefits.com,https://attachments.clickup.com/profilePicture...,901114339591,Data Inbox [GATEWAY],True,None,NaN,ADP-RMRWWC,Clients/Westside Women's Care-RMRWWC/File Feed...,NaN,None,TEST_WestsideWomenCareRMRWWC_Contribution_2026...,2026-09-18 23:00:00-06:00,NDT,2026-09-19 15:15:00-06:00,None,868k9unw0,group:75e50387-5a3b-47ec-968c-e75ec968acae
2,868m73b3f,None,0,TEST_WestsideWomenCare_RMRWWC_DemoElec_2026091...,,,2026-09-19 10:15:00-06:00,2026-09-19 10:15:00-06:00,None,NaT,False,[],[],[],[],[],None,None,None,None,None,None,None,[],[],[],9011096643,https://app.clickup.com/t/868m73b3f,create,new,sc901114339591_BKPkQ4Jh,#3db88b,open,0,75481693,rmrcloud,#5d4037,rmrcloud@rmrbenefits.com,https://attachments.clickup.com/profilePicture...,901114339591,Data Inbox [GATEWAY],True,None,NaN,ADP-RMRWWC,Clients/Westside Women's Care-RMRWWC/File Feed...,NaN,None,TEST_WestsideWomenCare_RMRWWC_DemoElec_2026091...,2026-09-18 23:00:00-06:00,FLEX,2026-09-19 15:15:00-06:00,None,868k9unw0,group:75e50387-5a3b-47ec-968c-e75ec968acae
3,868m738zg,None,0,Catholic_Community_Services_WW_RMRCCWW_DemoEle...,,,2026-09-19 09:44:00-06:00,2026-09-19 09:44:00-06:00,None,2026-09-19 09:44:00-06:00,False,[],[],[],[],[],None,None,None,None,None,None,None,[],[],[],9011096643,https://app.clickup.com/t/868m738zg,create,not to process,sc901114339591_bvGWkX7b,#87909e,done,8,75481693,rmrcloud,#5d4037,rmrcloud@rmrbenefits.com,https://attachments.clickup.com/profilePicture...,901114339591,Data Inbox [GATEWAY],True,None,NaN,jordan.suttle@rmrbenefits.com,Clients/Catholic Community Services WW-RMRCCWW...,NaN,None,Catholic_Community_Services_WW_RMRCCWW_DemoEle...,2026-09-18 23:00:00-06:00,FLEX,2026-09-18 22:08:00-06:00,None,868ft43m4,None
4,868m730dp,None,0,Design Workshop RMRDWI Contribution File 20260...,,,2026-09-19 08:07:00-06:00,2026-09-19 08:07:00-06:00,None,2026-09-19 08:07:00-06:00,False,[],[],[],[],[],None,None,None,None,None,None,None,[],[],[],9011096643,https://app.clickup.com/t/868m730dp,create,not to process,sc901114339591_bvGWkX7b,#87909e,done,8,75481693,rmrcloud,#5d4037,rmrcloud@rmrbenefits.com,https://attachments.clickup.com/profilePicture...,901114339591,Data Inbox [GATEWAY],True,None,NaN,kelsie.enos@rmrbenefits.com,Clients/Design Workshop-RMRDWI/File Feeds/Comp...,NaN,None,Design Workshop RMRDWI Contribution File 20260...,2026-09-18 23:00:00-06:00,NDT,2026-09-18 22:42:00-06:00,None,8687ad54g,None
...,...,...,

In [31]:
import json


with open("task_ids_to_delete.json", "w") as f:
    json.dump(task_ids_to_delete, f, indent=4)

In [ ]:
im